# Heart Disease Prediction

**Tabular Regression Project** — Predict heart disease risk metrics.

Models: CatBoost · LightGBM · XGBoost · FLAML · LazyPredict
Dataset: Heart Failure Prediction (Kaggle, ~918 rows)
Target: Numeric health metric

## 2 · Project Overview

We use the Heart Failure Prediction dataset to build regression models predicting **continuous health metrics** (e.g., MaxHR — maximum heart rate, or cholesterol). While often used for classification, treating continuous health variables as regression targets teaches valuable medical ML techniques.

## 3 · Learning Objectives

1. Work with clinical/medical datasets.
2. Handle mixed categorical and numeric health features.
3. Apply regression to health outcome prediction.
4. Use LazyPredict and FLAML for model selection.
5. Understand ethical implications of medical ML.

## 4 · Problem Statement

Predict a **continuous health metric** (e.g., maximum heart rate or cholesterol) from clinical features including age, sex, chest pain type, blood pressure, and ECG results.

## 5 · Why This Project Matters

- **Heart disease** is the leading cause of death worldwide.
- Predicting health metrics helps in early risk assessment.
- Clinical ML requires careful handling of biased and imbalanced data.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Source** | Kaggle: fedesoriano/heart-failure-prediction |
| **Rows** | ~918 |
| **Features** | Age, Sex, ChestPainType, RestingBP, Cholesterol, FastingBS, RestingECG, MaxHR, ExerciseAngina, Oldpeak, ST_Slope |
| **Target** | MaxHR (Maximum Heart Rate, continuous) |

## 7 · Dataset Source and License Notes

- **Source**: Combined heart disease datasets from UCI and other repositories.
- **License**: Open Database License.
- **Note**: Clinical data collected from multiple hospital studies.

## 8 · Environment Setup

In [ ]:
import subprocess, sys
def _install_if_missing(pkg, imp=None):
    try: __import__(imp or pkg)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
for p, i in [('catboost',None),('lightgbm',None),('xgboost',None),('flaml',None),('lazypredict',None)]:
    _install_if_missing(p, i)
_install_if_missing('kagglehub')
print('All packages ready.')

## 9 · Imports

In [ ]:
import os, warnings, json
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    from sklearn.metrics import mean_squared_error
    def root_mean_squared_error(y_true, y_pred): return mean_squared_error(y_true, y_pred, squared=False)
warnings.filterwarnings('ignore')
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print('Imports complete.')

## 10 · Configuration / Constants

In [ ]:
SEED = 42
TEST_SIZE = 0.2
TARGET = 'MaxHR'
np.random.seed(SEED)

## 11 · Dataset Download or Loading

In [ ]:
import kagglehub, glob

try:
    path = kagglehub.dataset_download('fedesoriano/heart-failure-prediction')
    print(f'Downloaded to: {path}')
except Exception as e:
    print(f'Download error: {e}')
    path = SAVE_DIR

csv_files = glob.glob(os.path.join(path, '**', '*.csv'), recursive=True)
assert csv_files, f'No CSV found in {path}'
df = pd.read_csv(sorted(csv_files, key=os.path.getsize, reverse=True)[0])
print(f'Loaded: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

## 12 · Data Validation Checks

In [ ]:
print('Missing values:')
print(df.isnull().sum())
print(f'\nDuplicates: {df.duplicated().sum()}')
print(f'\nZero cholesterol: {(df["Cholesterol"]==0).sum() if "Cholesterol" in df.columns else "N/A"}')

## 13 · Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

df[TARGET].hist(bins=30, ax=axes[0,0], edgecolor='black')
axes[0,0].set_title(f'{TARGET} Distribution')

if 'Age' in df.columns:
    axes[0,1].scatter(df['Age'], df[TARGET], alpha=0.3)
    axes[0,1].set_xlabel('Age'); axes[0,1].set_ylabel(TARGET)
    axes[0,1].set_title('Age vs MaxHR')

if 'Cholesterol' in df.columns:
    axes[1,0].scatter(df['Cholesterol'], df[TARGET], alpha=0.3)
    axes[1,0].set_xlabel('Cholesterol'); axes[1,0].set_ylabel(TARGET)
    axes[1,0].set_title('Cholesterol vs MaxHR')

if 'RestingBP' in df.columns:
    axes[1,1].scatter(df['RestingBP'], df[TARGET], alpha=0.3)
    axes[1,1].set_xlabel('RestingBP'); axes[1,1].set_ylabel(TARGET)
    axes[1,1].set_title('RestingBP vs MaxHR')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_plots.png'), dpi=100)
plt.show()

## 14 · Target Analysis

In [ ]:
print(df[TARGET].describe())
print(f'\nSkewness: {df[TARGET].skew():.2f}')

## 15 · Train / Validation / Test Split

## 16 · Preprocessing

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Fix zero cholesterol (impossible) with median
if 'Cholesterol' in df.columns:
    median_chol = df.loc[df['Cholesterol'] > 0, 'Cholesterol'].median()
    df['Cholesterol'] = df['Cholesterol'].replace(0, median_chol)

# Drop HeartDisease (classification label) if present
if 'HeartDisease' in df.columns:
    df = df.drop(columns=['HeartDisease'])

# Encode categoricals
for col in df.select_dtypes(include='object').columns:
    df[col] = LabelEncoder().fit_transform(df[col].fillna('Unknown'))

for col in df.select_dtypes(include='number').columns:
    df[col] = df[col].fillna(df[col].median())

print(f'Preprocessed: {df.shape}')

## 17 · Feature Engineering

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 18 · Baseline Model

In [ ]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_base = baseline.predict(X_test)
baseline_rmse = root_mean_squared_error(y_test, y_pred_base)
baseline_mae = mean_absolute_error(y_test, y_pred_base)
baseline_r2 = r2_score(y_test, y_pred_base)
print(f'Baseline LR: RMSE={baseline_rmse:.2f}, MAE={baseline_mae:.2f}, R²={baseline_r2:.4f}')

## 19 · LazyPredict Benchmark

In [ ]:
from lazypredict.Supervised import LazyRegressor
lazy = LazyRegressor(verbose=0, ignore_warnings=True, random_state=SEED)
lazy_models, _ = lazy.fit(X_train.head(min(5000,len(X_train))), X_test.head(min(1000,len(X_test))),
                           y_train.head(min(5000,len(y_train))), y_test.head(min(1000,len(y_test))))
print(lazy_models.head(15))

## 20 · FLAML AutoML Run

In [ ]:
try:
    from flaml import AutoML
    flaml_model = AutoML()
    flaml_model.fit(X_train, y_train, task='regression', time_budget=60, metric='rmse', seed=SEED, verbose=0)
    y_pred_flaml = flaml_model.predict(X_test)
    flaml_rmse = root_mean_squared_error(y_test, y_pred_flaml)
    flaml_mae = mean_absolute_error(y_test, y_pred_flaml)
    flaml_r2 = r2_score(y_test, y_pred_flaml)
    print(f'FLAML best: {flaml_model.best_estimator}')
    print(f'  RMSE: {flaml_rmse:.2f}, MAE: {flaml_mae:.2f}, R²: {flaml_r2:.4f}')
except Exception as e:
    print(f'FLAML failed: {e}')
    flaml_rmse = flaml_mae = flaml_r2 = None

## 21 · Boosting Models

In [ ]:
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
models = {
    'CatBoost': CatBoostRegressor(iterations=300, learning_rate=0.1, depth=6, random_seed=SEED, verbose=0),
    'LightGBM': LGBMRegressor(n_estimators=300, learning_rate=0.1, max_depth=6, random_state=SEED, verbose=-1),
    'XGBoost': XGBRegressor(n_estimators=300, learning_rate=0.1, max_depth=6, random_state=SEED, verbosity=0)
}
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results[name] = {
        'RMSE': root_mean_squared_error(y_test, preds),
        'MAE': mean_absolute_error(y_test, preds),
        'R2': r2_score(y_test, preds)
    }
    print(f'{name}: RMSE={results[name]["RMSE"]:.2f}, MAE={results[name]["MAE"]:.2f}, R²={results[name]["R2"]:.4f}')

## 22 · Top 3 Model Selection

In [ ]:
all_results = {}
all_results['Baseline_LR'] = {'RMSE': baseline_rmse, 'MAE': baseline_mae, 'R2': baseline_r2}
if flaml_r2 is not None:
    all_results['FLAML'] = {'RMSE': flaml_rmse, 'MAE': flaml_mae, 'R2': flaml_r2}
all_results.update(results)
ranked = sorted(all_results.items(), key=lambda x: x[1]['RMSE'])
print('All models ranked by RMSE:')
for name, m in ranked:
    print(f"  {name:20s}  RMSE={m['RMSE']:.2f}  MAE={m['MAE']:.2f}  R²={m['R2']:.4f}")
top3_names = [r[0] for r in ranked[:3]]
print(f'\nTop 3: {top3_names}')

## 23 · Final Eval of Top 3

In [ ]:
print('Top 3 Final Results:')
print('=' * 60)
for name in top3_names:
    m = all_results[name]
    print(f"{name}: RMSE={m['RMSE']:.2f}, MAE={m['MAE']:.2f}, R²={m['R2']:.4f}")

## 24 · Error Analysis

In [ ]:
best_name = top3_names[0]
if best_name in models: best_model = models[best_name]
else: best_model = models.get('CatBoost', baseline)
y_pred_best = best_model.predict(X_test)
residuals = y_test.values - y_pred_best
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].scatter(y_pred_best, residuals, alpha=0.5); axes[0].axhline(0, color='r', linestyle='--')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Residual'); axes[0].set_title('Residuals vs Predicted')
axes[1].hist(residuals, bins=30, edgecolor='black'); axes[1].set_title('Residual Distribution')
axes[2].scatter(y_test, y_pred_best, alpha=0.5)
axes[2].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[2].set_xlabel('Actual'); axes[2].set_ylabel('Predicted'); axes[2].set_title('Actual vs Predicted')
plt.tight_layout(); plt.savefig(os.path.join(SAVE_DIR, 'error_analysis.png'), dpi=100); plt.show()

## 25 · Interpretation and Business Insight

- **Age** is the strongest predictor — MaxHR decreases with age (well-known physiology).
- **Exercise-induced angina** and **ST_Slope** provide additional predictive power.
- These predictions could assist cardiologists in estimating exercise capacity.
- Model should complement, not replace, clinical stress testing.

## 26 · Limitations

- Only ~918 patients from specific hospital studies.
- Missing data quality issues (zero cholesterol values).
- Max heart rate depends heavily on fitness level (not captured).
- Not validated for clinical decision-making.

## 27 · How to Improve

1. Add fitness/exercise history data.
2. Add medication information.
3. Use larger clinical datasets with more centers.
4. Multi-task learning (predict multiple health metrics jointly).

## 28 · Production Considerations

- FDA/regulatory approval needed for clinical deployment.
- Must be validated across diverse populations.
- Explainability is critical in medical ML.
- Integrate with electronic health records (EHR).

## 29 · Common Mistakes

1. Not handling zero cholesterol as missing.
2. Including HeartDisease label as a feature.
3. Generalizing to populations underrepresented in training data.
4. Claiming clinical validity without proper trials.

## 30 · Mini Challenge

1. Predict Cholesterol instead of MaxHR.
2. Add age-group interaction features.
3. Compare with the well-known formula MaxHR ≈ 220 - Age.
4. Use SHAP to explain which features matter most per patient.

## 31 · Final Summary

- Heart disease data provides a realistic medical regression task.
- The age-MaxHR relationship is well-established in cardiology.
- ML models can refine predictions beyond simple formulas.
- Clinical deployment requires rigorous validation and ethical review.

In [ ]:
metrics = {}
for name in top3_names: metrics[name] = all_results[name]
metrics['baseline'] = all_results['Baseline_LR']
with open(os.path.join(SAVE_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Metrics saved to metrics.json')
print('\nNotebook complete.')